# GalaxyGAN (cWGAN-GP) on Google Colab

**Runtime:** `Runtime` → `Change runtime type` → **GPU**.

**Colab Pro — which GPU?** For this project (69×69 cWGAN-GP), VRAM is not the bottleneck.

- **L4** (if offered): best **speed vs. compute units** for medium jobs — pick this for long training runs when available.
- **A100**: fastest iterations if you have compute units to spend; overkill for memory but nice for **shorter wall-clock** and larger batches.
- **T4**: still fine for this model; use if L4/A100 are unavailable.

See [Colab pricing / runtimes](https://cloud.google.com/colab/pricing) for how your plan bills compute units.

**Project on Drive:** this notebook assumes the repo lives at  
`/content/drive/MyDrive/Galaxy GANS` (folder name with a space — fine in Python `pathlib`).

**First run:** downloads ~2.5 GB Galaxy10 HDF5 into `…/Galaxy GANS/data/` (Zenodo). Checkpoints go under `…/Galaxy GANS/runs/`.

In [ ]:
# Mount Drive + clone or update repo under MyDrive/Galaxy GANS (path can include spaces).
import os
import subprocess
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

REPO = Path("/content/drive/MyDrive/Galaxy GANS")
os.environ["GALAXYGAN_REPO"] = str(REPO)
REPO.mkdir(parents=True, exist_ok=True)

marker = REPO / "galaxygan" / "train.py"
if not marker.is_file():
    if any(REPO.iterdir()):
        raise RuntimeError(
            f"{REPO} exists but is not a GalaxyGAN clone. "
            "Empty that folder on Drive or pick a different REPO path."
        )
    subprocess.run(
        ["git", "clone", "https://github.com/ndhanrajani/GalaxyGAN.git", str(REPO)],
        check=True,
    )
else:
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=False)

os.chdir(REPO)
print("REPO =", REPO.resolve())
print("cwd  =", os.getcwd())

In [ ]:
# PyTorch (Colab usually has torch; this ensures torchvision + deps for our code)
!pip install -q h5py scikit-learn torchvision

import torch
print("cuda?", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")

In [ ]:
# Dataset (~2.54 GB) on Drive — survives Colab disconnects
import os
import subprocess
from pathlib import Path

REPO = Path(os.environ.get("GALAXYGAN_REPO", "/content/drive/MyDrive/Galaxy GANS"))
data_dir = REPO / "data"
data_dir.mkdir(parents=True, exist_ok=True)
h5 = data_dir / "Galaxy10_DECals.h5"
url = "https://zenodo.org/records/10845026/files/Galaxy10_DECals.h5"

if not h5.is_file():
    subprocess.run(["wget", "-O", str(h5), url], check=True)
else:
    print("Already present:", h5)

os.environ["GALAXY10_H5"] = str(h5)
print("GALAXY10_H5 =", h5)

In [ ]:
import os
from pathlib import Path

REPO = Path(os.environ.get("GALAXYGAN_REPO", "/content/drive/MyDrive/Galaxy GANS"))
OUT = str(REPO / "runs" / "colab_run")
os.makedirs(OUT, exist_ok=True)
print("OUT =", OUT)

_h5 = Path(os.environ.get("GALAXY10_H5", REPO / "data" / "Galaxy10_DECals.h5"))
print("GALAXY10_H5 (for train) =", _h5, "exists=", _h5.is_file())

In [ ]:
import os
import sys
from pathlib import Path

REPO = Path(os.environ.get("GALAXYGAN_REPO", "/content/drive/MyDrive/Galaxy GANS"))
os.chdir(REPO)

try:
    OUT
except NameError:
    OUT = str(REPO / "runs" / "colab_run")
    os.makedirs(OUT, exist_ok=True)

h5 = Path(os.environ.get("GALAXY10_H5", REPO / "data" / "Galaxy10_DECals.h5"))
if not h5.is_file():
    raise FileNotFoundError(
        f"Dataset missing: {h5}\nRun the download cell first (Zenodo ~2.5 GB)."
    )
if h5.stat().st_size < 1_000_000_000:
    raise RuntimeError(
        f"HDF5 looks truncated ({h5.stat().st_size} bytes). Delete it and re-run the download cell."
    )
os.environ["GALAXY10_H5"] = str(h5)

# Run in-process so Colab shows the *real* error (subprocess + check=True only shows CalledProcessError).
sys.argv = [
    "galaxygan.train",
    "--epochs",
    "3",
    "--batch-size",
    "64",
    "--out",
    OUT,
]
from galaxygan.train import main

main()